# 📈 Stock Portfolio Optimiser + Price Predictor

**Built as part of the RMIT Master of Analytics Program**

---

## Overview

This notebook demonstrates a complete, end-to-end financial analytics pipeline covering:

1. **Data Acquisition** — Real-time and historical stock data via `yfinance`
2. **Exploratory Data Analysis (EDA)** — Price trends, volume, rolling statistics
3. **Portfolio Construction** — Weighted allocation and blended return calculation
4. **Risk Metrics** — Sharpe ratio, volatility, Value at Risk, Beta, Max Drawdown
5. **Correlation Analysis** — Asset diversification and correlation heatmaps
6. **Forecasting with Facebook Prophet** — 30-day price predictions with uncertainty intervals
7. **Portfolio Optimisation** — Monte Carlo simulation and the efficient frontier
8. **Investment Insights** — Automated buy/hold/sell signal generation

---

**Stocks covered:** ASX (BHP.AX, CBA.AX, CSL.AX, WBC.AX) + Global (AAPL, MSFT, GOOGL, TSLA, JPM)  
**Tools:** Python 3, yfinance, pandas, numpy, Prophet, Plotly, scipy


In [ ]:
# ── 0. Install dependencies (run once) ─────────────────────────────────────
# Uncomment and run this cell if you haven't installed requirements yet
# !pip install yfinance pandas numpy prophet plotly scipy scikit-learn seaborn pyportfolioopt

In [ ]:
# ── 1. Imports ──────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import yfinance as yf
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from prophet import Prophet
from scipy import stats
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

print('✅ All libraries imported successfully')
print(f'Pandas: {pd.__version__} | NumPy: {np.__version__}')

---
## Section 1: Portfolio Configuration & Data Acquisition

We define a blended ASX + global equity portfolio and download 2 years of historical OHLCV data.


In [ ]:
# ── Portfolio Configuration ──────────────────────────────────────────────────
PORTFOLIO = {
    'BHP.AX': {'weight': 0.15, 'name': 'BHP Group',                    'sector': 'Materials'},
    'CBA.AX': {'weight': 0.15, 'name': 'Commonwealth Bank',            'sector': 'Financials'},
    'CSL.AX': {'weight': 0.10, 'name': 'CSL Limited',                  'sector': 'Healthcare'},
    'WBC.AX': {'weight': 0.05, 'name': 'Westpac Banking',              'sector': 'Financials'},
    'AAPL':   {'weight': 0.15, 'name': 'Apple Inc',                    'sector': 'Technology'},
    'MSFT':   {'weight': 0.15, 'name': 'Microsoft Corporation',        'sector': 'Technology'},
    'GOOGL':  {'weight': 0.10, 'name': 'Alphabet Inc',                 'sector': 'Technology'},
    'TSLA':   {'weight': 0.10, 'name': 'Tesla Inc',                    'sector': 'Consumer Disc.'},
    'JPM':    {'weight': 0.05, 'name': 'JPMorgan Chase',               'sector': 'Financials'},
}

TICKERS    = list(PORTFOLIO.keys())
WEIGHTS    = np.array([PORTFOLIO[t]['weight'] for t in TICKERS])
WEIGHTS    = WEIGHTS / WEIGHTS.sum()   # normalise to 1
PERIOD     = '2y'
RISK_FREE  = 0.04   # 4% annualised risk-free rate (approximate RBA cash rate)

print(f'Portfolio: {len(TICKERS)} stocks | Period: {PERIOD}')
pd.DataFrame({
    'Ticker': TICKERS,
    'Name': [PORTFOLIO[t]['name'] for t in TICKERS],
    'Sector': [PORTFOLIO[t]['sector'] for t in TICKERS],
    'Weight': [f'{w:.1%}' for w in WEIGHTS]
}).set_index('Ticker')

In [ ]:
# ── Download Historical Data ─────────────────────────────────────────────────
print('⏳ Downloading data from Yahoo Finance...')
stock_data = {}
failed = []

for ticker in TICKERS:
    try:
        df = yf.download(ticker, period=PERIOD, auto_adjust=True, progress=False)
        if df.empty or len(df) < 50:
            failed.append(ticker)
        else:
            stock_data[ticker] = df
            print(f'  ✅ {ticker:8s} — {len(df)} trading days loaded')
    except Exception as e:
        failed.append(ticker)
        print(f'  ❌ {ticker:8s} — Failed: {e}')

VALID_TICKERS = [t for t in TICKERS if t in stock_data]
VALID_WEIGHTS = np.array([PORTFOLIO[t]['weight'] for t in VALID_TICKERS])
VALID_WEIGHTS = VALID_WEIGHTS / VALID_WEIGHTS.sum()

print(f'\n✅ {len(VALID_TICKERS)}/{len(TICKERS)} tickers loaded successfully')
if failed:
    print(f'❌ Failed: {failed}')

---
## Section 2: Exploratory Data Analysis (EDA)

We examine price trends, trading volume, and rolling statistics for each asset.

In [ ]:
# ── Closing Price Summary Statistics ────────────────────────────────────────
close_prices = pd.DataFrame({t: stock_data[t]['Close'] for t in VALID_TICKERS}).dropna()

print('=== Closing Price Summary ===')
summary = close_prices.describe().T
summary.index.name = 'Ticker'
summary = summary[['mean', 'std', 'min', 'max']].rename(columns={
    'mean': 'Mean Price', 'std': 'Std Dev', 'min': 'Min', 'max': 'Max'
})
summary['Range (%)'] = ((summary['Max'] - summary['Min']) / summary['Min'] * 100).map('{:.1f}%'.format)
print(summary.round(2).to_string())

In [ ]:
# ── Normalised Price Chart (Base = 100) ─────────────────────────────────────
normalised = close_prices / close_prices.iloc[0] * 100

fig = go.Figure()
colors = px.colors.qualitative.Plotly
for i, t in enumerate(VALID_TICKERS):
    fig.add_trace(go.Scatter(
        x=normalised.index, y=normalised[t],
        name=f'{t} ({PORTFOLIO[t]["name"]})',
        line=dict(color=colors[i % len(colors)])
    ))

fig.update_layout(
    title='Normalised Price Performance (Base = 100)',
    xaxis_title='Date', yaxis_title='Normalised Price',
    template='plotly_white', height=480,
    legend=dict(orientation='h', yanchor='bottom', y=1.01, xanchor='right', x=1)
)
fig.show()

In [ ]:
# ── Candlestick Chart for a Single Stock ────────────────────────────────────
DETAIL_TICKER = 'AAPL'
df_detail = stock_data[DETAIL_TICKER].tail(180)   # last 6 months

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    row_heights=[0.75, 0.25], vertical_spacing=0.04,
                    subplot_titles=(f'{DETAIL_TICKER} — OHLC (Last 6 Months)', 'Volume'))

fig.add_trace(go.Candlestick(
    x=df_detail.index,
    open=df_detail['Open'], high=df_detail['High'],
    low=df_detail['Low'],   close=df_detail['Close'],
    increasing_line_color='#10B981', decreasing_line_color='#EF4444',
    name='OHLC'
), row=1, col=1)

# Add 20-day and 50-day moving averages
fig.add_trace(go.Scatter(
    x=df_detail.index, y=df_detail['Close'].rolling(20).mean(),
    name='MA20', line=dict(color='#F59E0B', width=1.5)
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=df_detail.index, y=df_detail['Close'].rolling(50).mean(),
    name='MA50', line=dict(color='#8B5CF6', width=1.5)
), row=1, col=1)

fig.add_trace(go.Bar(
    x=df_detail.index, y=df_detail['Volume'],
    marker_color='#93C5FD', opacity=0.7, name='Volume'
), row=2, col=1)

fig.update_layout(height=500, template='plotly_white',
                  xaxis_rangeslider_visible=False)
fig.show()

In [ ]:
# ── Bollinger Bands ──────────────────────────────────────────────────────────
window = 20
close = df_detail['Close']
ma   = close.rolling(window).mean()
std  = close.rolling(window).std()
upper_band = ma + 2 * std
lower_band = ma - 2 * std

fig = go.Figure()
fig.add_trace(go.Scatter(x=close.index, y=upper_band, name='Upper Band (2σ)',
                         line=dict(color='#EF4444', dash='dash')))
fig.add_trace(go.Scatter(x=close.index, y=lower_band, name='Lower Band (2σ)',
                         line=dict(color='#10B981', dash='dash'),
                         fill='tonexty', fillcolor='rgba(37,99,235,0.08)'))
fig.add_trace(go.Scatter(x=close.index, y=ma, name=f'MA{window}',
                         line=dict(color='#2563EB', width=2)))
fig.add_trace(go.Scatter(x=close.index, y=close, name='Close Price',
                         line=dict(color='black', width=1.5)))

fig.update_layout(title=f'{DETAIL_TICKER} — Bollinger Bands (20-day, 2σ)',
                  xaxis_title='Date', yaxis_title='Price',
                  template='plotly_white', height=400)
fig.show()

---
## Section 3: Return Analysis

Daily returns, cumulative performance, and distribution characteristics.

In [ ]:
# ── Daily & Cumulative Returns ───────────────────────────────────────────────
returns_df = close_prices.pct_change().dropna()

# Blended portfolio returns (weighted sum)
port_returns = returns_df[VALID_TICKERS].dot(VALID_WEIGHTS)
port_cum = (1 + port_returns).cumprod() - 1

print('=== Annualised Return Summary ===')
for t, w in zip(VALID_TICKERS, VALID_WEIGHTS):
    ann_r = (1 + returns_df[t].mean()) ** 252 - 1
    print(f'  {t:8s}: {ann_r:+.1%}  (weight: {w:.1%})')
ann_port = (1 + port_returns.mean()) ** 252 - 1
print(f'\n  Portfolio: {ann_port:+.1%} (blended)')

In [ ]:
# ── Cumulative Return Chart ──────────────────────────────────────────────────
fig = go.Figure()
colors = px.colors.qualitative.Plotly

for i, t in enumerate(VALID_TICKERS):
    cum_ret = (1 + returns_df[t]).cumprod() - 1
    fig.add_trace(go.Scatter(
        x=cum_ret.index, y=cum_ret,
        name=t, line=dict(color=colors[i % len(colors)]), opacity=0.75
    ))

fig.add_trace(go.Scatter(
    x=port_cum.index, y=port_cum,
    name='Portfolio (Blended)', line=dict(color='black', width=3, dash='dash')
))

fig.update_layout(
    title='Cumulative Returns (All Assets + Blended Portfolio)',
    xaxis_title='Date', yaxis_title='Cumulative Return',
    yaxis_tickformat='.0%', template='plotly_white', height=460,
    legend=dict(orientation='h', yanchor='bottom', y=1.01, xanchor='right', x=1)
)
fig.show()

In [ ]:
# ── Return Distribution (QQ Plot + Histogram) ────────────────────────────────
ticker_to_plot = 'AAPL'
r = returns_df[ticker_to_plot].dropna()

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Return Histogram', 'Q-Q Plot (Normal Reference)'])

# Histogram
fig.add_trace(go.Histogram(
    x=r, nbinsx=60, histnorm='probability density',
    name='Daily Returns', marker_color='#2563EB', opacity=0.75
), row=1, col=1)

# Normal PDF overlay
x_range = np.linspace(r.min(), r.max(), 200)
normal_pdf = stats.norm.pdf(x_range, r.mean(), r.std())
fig.add_trace(go.Scatter(
    x=x_range, y=normal_pdf, name='Normal PDF',
    line=dict(color='#EF4444', width=2)
), row=1, col=1)

# Q-Q plot
qq = stats.probplot(r, dist='norm')
fig.add_trace(go.Scatter(
    x=qq[0][0], y=qq[0][1], mode='markers',
    marker=dict(color='#2563EB', size=4), name='Quantiles'
), row=1, col=2)
fig.add_trace(go.Scatter(
    x=qq[0][0], y=qq[1][0] + qq[1][1] * qq[0][0],
    mode='lines', name='Normal Line',
    line=dict(color='#EF4444', width=2)
), row=1, col=2)

fig.update_layout(title=f'{ticker_to_plot} Return Distribution Analysis',
                  template='plotly_white', height=380, showlegend=False)
fig.show()

# Summary stats
print(f'Skewness : {r.skew():.4f}  (negative = left tail, market crash risk)')
print(f'Kurtosis : {r.kurtosis():.4f}  (>0 = fat tails vs normal distribution)')
_, pval = stats.shapiro(r.sample(min(len(r), 500), random_state=42))
print(f'Shapiro-Wilk p-value: {pval:.4f}  ({"Non-normal" if pval < 0.05 else "Normal"} distribution at 5%)')

---
## Section 4: Portfolio Risk Metrics

Computing Sharpe ratio, Value at Risk, Beta, Alpha, and Maximum Drawdown.

In [ ]:
# ── Risk Metric Functions ────────────────────────────────────────────────────
def annualised_return(returns): return (1 + returns.mean()) ** 252 - 1
def annualised_vol(returns):    return returns.std() * np.sqrt(252)
def sharpe(returns, rf=0.04):   return (annualised_return(returns) - rf) / annualised_vol(returns)
def var_95(returns):            return float(np.percentile(returns.dropna(), 5))
def cvar_95(returns):
    v = var_95(returns)
    return returns[returns <= v].mean()

def max_drawdown(prices):
    roll_max = prices.cummax()
    dd = (prices - roll_max) / roll_max
    return dd.min()

def beta(asset_ret, market_ret):
    aligned = pd.DataFrame({'a': asset_ret, 'm': market_ret}).dropna()
    cov_mat = np.cov(aligned['a'], aligned['m'])
    return cov_mat[0, 1] / cov_mat[1, 1] if cov_mat[1, 1] != 0 else 1.0

market_proxy = returns_df.mean(axis=1)   # equally-weighted proxy for the market

# ── Risk Table ───────────────────────────────────────────────────────────────
rows = []
for t in VALID_TICKERS:
    r  = returns_df[t]
    pr = close_prices[t]
    b  = beta(r, market_proxy)
    ann_r = annualised_return(r)
    alpha = ann_r - (RISK_FREE + b * (annualised_return(market_proxy) - RISK_FREE))
    rows.append({
        'Ticker':         t,
        'Ann. Return':    f'{ann_r:.1%}',
        'Ann. Volatility':f'{annualised_vol(r):.1%}',
        'Sharpe Ratio':   f'{sharpe(r):.2f}',
        'VaR (95%)':      f'{var_95(r):.2%}',
        'CVaR (95%)':     f'{cvar_95(r):.2%}',
        'Max Drawdown':   f'{max_drawdown(pr):.1%}',
        'Beta':           f'{b:.2f}',
        'Alpha':          f'{alpha:.1%}',
        'Skewness':       f'{r.skew():.2f}',
        'Kurtosis':       f'{r.kurtosis():.2f}',
    })

risk_df = pd.DataFrame(rows).set_index('Ticker')
print('=== Individual Stock Risk Metrics ===')
print(risk_df.to_string())

# Portfolio-level
print('\n=== Blended Portfolio Metrics ===')
port_vol = annualised_vol(port_returns)
port_sr  = sharpe(port_returns)
port_var = var_95(port_returns)
port_cvar = cvar_95(port_returns)
port_dd  = max_drawdown((1 + port_returns).cumprod())
print(f'  Annualised Return   : {ann_port:.2%}')
print(f'  Annualised Volatility: {port_vol:.2%}')
print(f'  Sharpe Ratio        : {port_sr:.2f}')
print(f'  VaR (95%)           : {port_var:.2%}')
print(f'  CVaR (95%)          : {port_cvar:.2%}')
print(f'  Maximum Drawdown    : {port_dd:.2%}')

In [ ]:
# ── Drawdown Chart ───────────────────────────────────────────────────────────
port_price = (1 + port_returns).cumprod()
roll_max   = port_price.cummax()
drawdown   = (port_price - roll_max) / roll_max

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=drawdown.index, y=drawdown,
    fill='tozeroy', fillcolor='rgba(239,68,68,0.2)',
    line=dict(color='#EF4444'), name='Drawdown'
))
fig.update_layout(
    title='Portfolio Drawdown Over Time',
    xaxis_title='Date', yaxis_title='Drawdown',
    yaxis_tickformat='.0%', template='plotly_white', height=350
)
fig.show()

---
## Section 5: Correlation Analysis

Portfolio diversification depends on low or negative correlations between assets.

In [ ]:
# ── Correlation Heatmap ──────────────────────────────────────────────────────
corr = returns_df[VALID_TICKERS].corr()

fig = px.imshow(
    corr, text_auto='.2f', aspect='auto',
    color_continuous_scale='RdBu_r', zmin=-1, zmax=1,
    title='Asset Return Correlation Matrix'
)
fig.update_layout(height=480, template='plotly_white')
fig.show()

# High-correlation pairs
print('High-correlation pairs (|r| > 0.6):')
ticks = VALID_TICKERS
for i in range(len(ticks)):
    for j in range(i+1, len(ticks)):
        c = corr.iloc[i,j]
        if abs(c) > 0.6:
            print(f'  {ticks[i]} ↔ {ticks[j]}: {c:.2f}')

In [ ]:
# ── Rolling 90-Day Correlation ────────────────────────────────────────────────
t1, t2 = 'AAPL', 'MSFT'
rolling_corr = returns_df[t1].rolling(90).corr(returns_df[t2])

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=rolling_corr.index, y=rolling_corr,
    mode='lines', line=dict(color='#2563EB'), name=f'{t1} vs {t2}'
))
fig.add_hline(y=0, line_dash='dash', line_color='gray')
fig.add_hrect(y0=0.7, y1=1.0, fillcolor='red', opacity=0.07, line_width=0)
fig.add_hrect(y0=-1.0, y1=-0.7, fillcolor='green', opacity=0.07, line_width=0)
fig.update_layout(
    title=f'Rolling 90-Day Correlation: {t1} vs {t2}',
    xaxis_title='Date', yaxis_title='Correlation',
    template='plotly_white', height=340
)
fig.show()

---
## Section 6: Price Forecasting with Facebook Prophet

Prophet is a decomposable additive model developed by Meta for business time-series forecasting.
It handles trend, weekly seasonality, and yearly seasonality automatically.

In [ ]:
# ── Prophet Forecast Function ────────────────────────────────────────────────
def prophet_forecast(price_df: pd.DataFrame, horizon: int = 30,
                     ticker: str = '') -> tuple:
    """
    Fit a Prophet model and return (model, forecast_df).
    """
    df_p = price_df[['Close']].copy().reset_index()
    df_p.columns = ['ds', 'y']
    df_p['ds'] = pd.to_datetime(df_p['ds']).dt.tz_localize(None)
    df_p = df_p.dropna()

    model = Prophet(
        daily_seasonality  = False,
        weekly_seasonality = True,
        yearly_seasonality = True,
        changepoint_prior_scale = 0.05,
        seasonality_mode   = 'multiplicative',
        interval_width     = 0.95
    )
    model.fit(df_p)

    future   = model.make_future_dataframe(periods=horizon, freq='B')
    forecast = model.predict(future)
    return model, forecast


# ── Run Forecast ─────────────────────────────────────────────────────────────
FORECAST_TICKER  = 'AAPL'
FORECAST_HORIZON = 30   # business days

print(f'⏳ Running Prophet forecast for {FORECAST_TICKER} ({FORECAST_HORIZON}-day)...')
model, forecast = prophet_forecast(
    stock_data[FORECAST_TICKER], horizon=FORECAST_HORIZON, ticker=FORECAST_TICKER
)
print('✅ Forecast complete')

# Show tail of forecast
last_date = stock_data[FORECAST_TICKER].index[-1]
future_fc = forecast[forecast['ds'] > pd.Timestamp(last_date).tz_localize(None)]
print(f'\nForecast tail (next {min(7, FORECAST_HORIZON)} business days):')
display_cols = ['ds', 'yhat', 'yhat_lower', 'yhat_upper']
print(future_fc[display_cols].head(7).rename(columns={
    'ds': 'Date', 'yhat': 'Predicted', 'yhat_lower': 'Lower 95%', 'yhat_upper': 'Upper 95%'
}).round(2).to_string(index=False))

In [ ]:
# ── Forecast Visualisation ───────────────────────────────────────────────────
price_df = stock_data[FORECAST_TICKER]

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    row_heights=[0.75, 0.25], vertical_spacing=0.05,
                    subplot_titles=(f'{FORECAST_TICKER} — Price + Prophet Forecast', 'Volume'))

# Candlestick
fig.add_trace(go.Candlestick(
    x=price_df.index,
    open=price_df['Open'], high=price_df['High'],
    low=price_df['Low'],   close=price_df['Close'],
    increasing_line_color='#10B981', decreasing_line_color='#EF4444',
    name='OHLC'
), row=1, col=1)

# Forecast line
fc_future = forecast[forecast['ds'] > pd.Timestamp(price_df.index[-1]).tz_localize(None)]
fig.add_trace(go.Scatter(
    x=fc_future['ds'], y=fc_future['yhat'],
    name='Forecast', line=dict(color='#2563EB', dash='dash', width=2)
), row=1, col=1)

# Confidence interval
fig.add_trace(go.Scatter(
    x=pd.concat([fc_future['ds'], fc_future['ds'][::-1]]),
    y=pd.concat([fc_future['yhat_upper'], fc_future['yhat_lower'][::-1]]),
    fill='toself', fillcolor='rgba(37,99,235,0.1)',
    line=dict(color='rgba(0,0,0,0)'), name='95% CI'
), row=1, col=1)

# Volume
fig.add_trace(go.Bar(
    x=price_df.index, y=price_df['Volume'],
    marker_color='#93C5FD', opacity=0.7, name='Volume'
), row=2, col=1)

fig.update_layout(height=540, template='plotly_white',
                  xaxis_rangeslider_visible=False)
fig.show()

In [ ]:
# ── Prophet Components Plot ──────────────────────────────────────────────────
# Shows trend, weekly and yearly seasonality decomposition
from prophet.plot import plot_components_plotly
fig_components = plot_components_plotly(model, forecast)
fig_components.update_layout(height=600, title=f'{FORECAST_TICKER} — Prophet Components')
fig_components.show()

In [ ]:
# ── Forecast Multiple Tickers ─────────────────────────────────────────────────
print('Running forecasts for all portfolio stocks...')
forecasts = {}
for t in VALID_TICKERS:
    try:
        _, fc = prophet_forecast(stock_data[t], horizon=30)
        forecasts[t] = fc
        last_p = float(stock_data[t]['Close'].iloc[-1])
        last_ts = pd.Timestamp(stock_data[t].index[-1]).tz_localize(None)
        fc_future = fc[fc['ds'] > last_ts]
        pred_30  = float(fc_future['yhat'].iloc[-1]) if not fc_future.empty else last_p
        change   = (pred_30 - last_p) / last_p
        print(f'  {t:8s}: Current ${last_p:.2f} → 30-day forecast ${pred_30:.2f}  ({change:+.1%})')
    except Exception as e:
        print(f'  {t:8s}: Forecast failed — {e}')

---
## Section 7: Portfolio Optimisation — Monte Carlo Efficient Frontier

We simulate 10,000 random portfolios and identify the optimal weights for maximising the Sharpe ratio.

In [ ]:
# ── Monte Carlo Simulation ───────────────────────────────────────────────────
N_SIMULATIONS = 10_000
n_assets = len(VALID_TICKERS)
np.random.seed(42)

results = []
weight_matrix = []

for _ in range(N_SIMULATIONS):
    w = np.random.dirichlet(np.ones(n_assets))
    pr = returns_df[VALID_TICKERS].dot(w)
    ann_r = annualised_return(pr)
    ann_v = annualised_vol(pr)
    sr    = (ann_r - RISK_FREE) / ann_v if ann_v > 0 else 0
    results.append({'Return': ann_r, 'Volatility': ann_v, 'Sharpe': sr})
    weight_matrix.append(w)

mc_df = pd.DataFrame(results)
weight_matrix = np.array(weight_matrix)

best_idx   = mc_df['Sharpe'].idxmax()
minvol_idx = mc_df['Volatility'].idxmin()

best_weights  = weight_matrix[best_idx]
minvol_weights = weight_matrix[minvol_idx]

print(f'✅ {N_SIMULATIONS:,} simulations complete')
print(f'\n=== Maximum Sharpe Portfolio ===')
print(f'  Return: {mc_df.loc[best_idx, "Return"]:.2%}')
print(f'  Volatility: {mc_df.loc[best_idx, "Volatility"]:.2%}')
print(f'  Sharpe Ratio: {mc_df.loc[best_idx, "Sharpe"]:.2f}')
print(f'  Weights: {dict(zip(VALID_TICKERS, best_weights.round(3)))}')

print(f'\n=== Minimum Volatility Portfolio ===')
print(f'  Return: {mc_df.loc[minvol_idx, "Return"]:.2%}')
print(f'  Volatility: {mc_df.loc[minvol_idx, "Volatility"]:.2%}')
print(f'  Sharpe Ratio: {mc_df.loc[minvol_idx, "Sharpe"]:.2f}')
print(f'  Weights: {dict(zip(VALID_TICKERS, minvol_weights.round(3)))}')

print(f'\n=== Current Portfolio ===')
print(f'  Return: {ann_port:.2%}')
print(f'  Volatility: {port_vol:.2%}')
print(f'  Sharpe Ratio: {port_sr:.2f}')

In [ ]:
# ── Efficient Frontier Chart ──────────────────────────────────────────────────
fig = go.Figure()

# All simulated portfolios (coloured by Sharpe)
fig.add_trace(go.Scatter(
    x=mc_df['Volatility'], y=mc_df['Return'],
    mode='markers',
    marker=dict(size=3, color=mc_df['Sharpe'], colorscale='Viridis',
                showscale=True, colorbar=dict(title='Sharpe', len=0.6)),
    name='Simulated Portfolios', opacity=0.6
))

# Current portfolio
fig.add_trace(go.Scatter(
    x=[port_vol], y=[ann_port], mode='markers',
    marker=dict(size=16, color='#EF4444', symbol='star'),
    name='Current Portfolio'
))

# Max Sharpe
fig.add_trace(go.Scatter(
    x=[mc_df.loc[best_idx, 'Volatility']], y=[mc_df.loc[best_idx, 'Return']],
    mode='markers',
    marker=dict(size=16, color='#10B981', symbol='diamond'),
    name='Max Sharpe'
))

# Min Volatility
fig.add_trace(go.Scatter(
    x=[mc_df.loc[minvol_idx, 'Volatility']], y=[mc_df.loc[minvol_idx, 'Return']],
    mode='markers',
    marker=dict(size=16, color='#F59E0B', symbol='triangle-up'),
    name='Min Volatility'
))

fig.update_layout(
    title=f'Efficient Frontier — Monte Carlo ({N_SIMULATIONS:,} Simulations)',
    xaxis_title='Annualised Volatility', yaxis_title='Annualised Return',
    xaxis_tickformat='.0%', yaxis_tickformat='.0%',
    template='plotly_white', height=500
)
fig.show()

In [ ]:
# ── Optimal Weight Comparison ─────────────────────────────────────────────────
comparison_df = pd.DataFrame({
    'Ticker': VALID_TICKERS,
    'Current': [f'{w:.1%}' for w in VALID_WEIGHTS],
    'Max Sharpe': [f'{w:.1%}' for w in best_weights],
    'Min Volatility': [f'{w:.1%}' for w in minvol_weights]
}).set_index('Ticker')

print('=== Portfolio Weight Comparison ===')
print(comparison_df.to_string())

# Visualise weight differences
fig = go.Figure()
x = VALID_TICKERS
fig.add_trace(go.Bar(name='Current', x=x, y=VALID_WEIGHTS, marker_color='#2563EB'))
fig.add_trace(go.Bar(name='Max Sharpe', x=x, y=best_weights, marker_color='#10B981'))
fig.add_trace(go.Bar(name='Min Volatility', x=x, y=minvol_weights, marker_color='#F59E0B'))
fig.update_layout(
    barmode='group', title='Weight Comparison: Current vs Optimised',
    xaxis_title='Ticker', yaxis_title='Weight',
    yaxis_tickformat='.0%', template='plotly_white', height=380
)
fig.show()

---
## Section 8: Investment Signals & Summary

Automated signal generation based on quantitative criteria (Sharpe, return, drawdown).

In [ ]:
# ── Signal Generation Logic ──────────────────────────────────────────────────
def generate_signal(sr, ann_r, max_dd):
    score = 0
    if sr > 1.5:    score += 2
    elif sr > 0.8:  score += 1
    elif sr < 0:    score -= 2

    if ann_r > 0.15:  score += 2
    elif ann_r > 0.05: score += 1
    elif ann_r < -0.05: score -= 2

    if max_dd > -0.10: score += 1
    elif max_dd < -0.25: score -= 1

    if score >= 3:   return 'BUY',  score
    elif score >= 0: return 'HOLD', score
    else:            return 'SELL', score

# ── Build Signal Table ────────────────────────────────────────────────────────
signal_rows = []
for t in VALID_TICKERS:
    r   = returns_df[t]
    pr  = close_prices[t]
    ann_r = annualised_return(r)
    ann_v = annualised_vol(r)
    sr    = sharpe(r)
    md    = max_drawdown(pr)
    var_v = var_95(r)
    sig, score = generate_signal(sr, ann_r, md)
    current_price = float(pr.iloc[-1])

    # 30-day forecast (if available)
    pred_change = None
    if t in forecasts:
        fc = forecasts[t]
        last_ts = pd.Timestamp(stock_data[t].index[-1]).tz_localize(None)
        fc_fut = fc[fc['ds'] > last_ts]
        if not fc_fut.empty:
            pred_30 = float(fc_fut['yhat'].iloc[-1])
            pred_change = (pred_30 - current_price) / current_price

    signal_rows.append({
        'Ticker':        t,
        'Name':          PORTFOLIO[t]['name'],
        'Sector':        PORTFOLIO[t]['sector'],
        'Signal':        sig,
        'Score':         score,
        'Current Price': f'${current_price:.2f}',
        '30D Forecast':  f'{pred_change:+.1%}' if pred_change is not None else 'N/A',
        'Ann. Return':   f'{ann_r:.1%}',
        'Volatility':    f'{ann_v:.1%}',
        'Sharpe Ratio':  f'{sr:.2f}',
        'VaR (95%)':     f'{var_v:.2%}',
        'Max Drawdown':  f'{md:.1%}',
    })

signals_df = pd.DataFrame(signal_rows).set_index('Ticker')
print('=== Investment Signal Summary ===')
print(signals_df.to_string())

In [ ]:
# ── Signal Distribution Chart ─────────────────────────────────────────────────
signal_counts = signals_df['Signal'].value_counts().reset_index()
signal_counts.columns = ['Signal', 'Count']
colors_map = {'BUY': '#10B981', 'HOLD': '#F59E0B', 'SELL': '#EF4444'}

fig = px.bar(
    signal_counts, x='Signal', y='Count',
    color='Signal', color_discrete_map=colors_map,
    title='Portfolio Signal Distribution',
    text='Count'
)
fig.update_layout(template='plotly_white', height=330, showlegend=False)
fig.show()

# Summary
buys  = (signals_df['Signal'] == 'BUY').sum()
holds = (signals_df['Signal'] == 'HOLD').sum()
sells = (signals_df['Signal'] == 'SELL').sum()
print(f'\nBUY:  {buys} stocks ({buys/len(VALID_TICKERS):.0%})')
print(f'HOLD: {holds} stocks ({holds/len(VALID_TICKERS):.0%})')
print(f'SELL: {sells} stocks ({sells/len(VALID_TICKERS):.0%})')

In [ ]:
# ── Final Executive Summary ───────────────────────────────────────────────────
print('=' * 60)
print('  PORTFOLIO EXECUTIVE SUMMARY')
print('  RMIT Master of Analytics — Stock Portfolio Optimiser')
print('=' * 60)
print(f'  Analysis Date     : {datetime.today().strftime("%B %d, %Y")}')
print(f'  Stocks Analysed   : {len(VALID_TICKERS)} ({PERIOD} data)')
print(f'  Forecast Horizon  : {FORECAST_HORIZON} business days')
print(f'  Risk-Free Rate    : {RISK_FREE:.1%} p.a.')
print()
print('  --- Portfolio Performance ---')
print(f'  Annualised Return : {ann_port:.2%}')
print(f'  Annualised Vol    : {port_vol:.2%}')
print(f'  Sharpe Ratio      : {port_sr:.2f}')
print(f'  Maximum Drawdown  : {port_dd:.2%}')
print(f'  VaR (95%)         : {port_var:.2%}')
print(f'  CVaR (95%)        : {port_cvar:.2%}')
print()
print('  --- Optimisation Targets ---')
print(f'  Max Sharpe Ratio  : {mc_df.loc[best_idx, "Sharpe"]:.2f}')
print(f'  Min Volatility    : {mc_df.loc[minvol_idx, "Volatility"]:.2%}')
print()
print('  --- Investment Signals ---')
print(f'  BUY  : {buys} stocks')
print(f'  HOLD : {holds} stocks')
print(f'  SELL : {sells} stocks')
print()
if port_sr > 1.0:
    print('  RECOMMENDATION: Portfolio is performing well above risk-adjusted')
    print('  targets. Maintain allocations and review quarterly.')
else:
    print('  RECOMMENDATION: Portfolio Sharpe below 1.0. Consider rebalancing')
    print('  toward Max Sharpe optimal weights identified via Monte Carlo.')
print('=' * 60)
print('  DISCLAIMER: For educational purposes only. Not financial advice.')
print('=' * 60)

---
## Appendix: Methodology Notes

### Sharpe Ratio
$$\text{Sharpe} = \frac{R_p - R_f}{\sigma_p}$$
Where $R_p$ = annualised portfolio return, $R_f$ = risk-free rate, $\sigma_p$ = annualised volatility.

### Value at Risk (VaR)
Computed as the 5th percentile of the historical daily return distribution (historical simulation method).

### Conditional VaR (CVaR / Expected Shortfall)
The mean of returns that fall below the VaR threshold — a coherent risk measure that captures tail risk.

### Prophet Forecasting
Facebook Prophet decomposes the time series into:
- **Trend**: Piecewise linear or logistic growth
- **Seasonality**: Fourier series for weekly and yearly cycles
- **Residuals**: Unexplained variation

Uncertainty intervals are computed via Monte Carlo sampling of model posteriors.

### Monte Carlo Optimisation
10,000 random weight vectors are drawn from a Dirichlet distribution, ensuring all weights are positive and sum to 1. Each portfolio's Sharpe ratio is computed and the globally optimal set is identified.

---
*Built for RMIT Master of Analytics — Data Science Capstone Project*
